# Kernel Approximation pro redukci dimenzionality

## Co je Kernel Approximation?

Kernel Approximation je sada technik, které umožňují efektivně aproximovat jádrové funkce (kernel functions) často používané v metodách strojového učení. Místo explicitního výpočtu jádrových matic, které jsou výpočetně náročné pro velké datové sady, kernel approximation vytváří explicitní mapování do nového prostoru příznaků, které aproximuje jádrovou transformaci.

### Klíčové vlastnosti Kernel Approximation:

- **Efektivita**: Snižuje výpočetní složitost z O(n²) na O(n)
- **Škálovatelnost**: Umožňuje použití jádrových metod na velké datové sady
- **Nelineární transformace**: Zachovává schopnost modelovat nelineární vztahy
- **Explicitní mapování**: Poskytuje nový prostor příznaků, který lze použít s lineárními algoritmy

### Kdy použít Kernel Approximation:

1. **Velké datové sady**: Když máte příliš mnoho vzorků pro tradiční jádrové metody
2. **Omezené výpočetní zdroje**: Když nemáte dostatek paměti nebo času pro výpočet plné jádrové matice
3. **Online učení**: Pro aplikace vyžadující inkrementální aktualizace
4. **Jako předstupeň pro lineární metody**: Pro kombinaci výhod nelineárních jader s lineárními algoritmy

### Hlavní metody Kernel Approximation v scikit-learn:

1. **Nystroem**: Aproximuje jádrovou matici pomocí podvzorkování
2. **RBFSampler**: Aproximuje RBF jádro pomocí náhodných Fourierových vlastností
3. **SkewedChi2Sampler**: Aproximuje chi-squared jádro pomocí náhodných vlastností
4. **AdditiveChi2Sampler**: Aproximuje aditivní chi-squared jádro pomocí specifického mapování vlastností

V tomto notebooku se zaměříme především na metody Nystroem a RBFSampler jako nejčastěji používané techniky kernel approximation pro redukci dimenzionality.

In [ ]:
# Import potřebných knihoven
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pandas as pd
from scipy.spatial.distance import pdist, squareform

from sklearn.kernel_approximation import Nystroem, RBFSampler, SkewedChi2Sampler, AdditiveChi2Sampler
from sklearn.datasets import make_circles, make_moons, load_digits, fetch_openml, make_swiss_roll
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, pairwise_distances
from sklearn.decomposition import PCA, KernelPCA
from sklearn.manifold import TSNE, Isomap

import warnings
warnings.filterwarnings('ignore')

# Pro lepší vizualizaci
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
np.random.seed(42)

## 1. Základy Kernel Approximation

Nejprve se podíváme na základy kernel approximation a jak funguje. Začneme s jednoduchým příkladem, který ilustruje, jak kernel approximation transformuje data.

In [ ]:
# Vytvoříme jednoduchý nelineárně oddělitelný dataset - dvě soustředné kružnice
X, y = make_circles(n_samples=1000, factor=0.3, noise=0.05, random_state=42)

# Vizualizace původních dat
plt.figure(figsize=(10, 8))
plt.scatter(X[y==0, 0], X[y==0, 1], color='blue', alpha=0.7, label='Třída 0')
plt.scatter(X[y==1, 0], X[y==1, 1], color='red', alpha=0.7, label='Třída 1')
plt.title('Původní data - dvě kružnice', fontsize=15)
plt.xlabel('Příznak 1', fontsize=12)
plt.ylabel('Příznak 2', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Aplikace různých metod kernel approximation
# 1. Nystroem metoda
nystroem = Nystroem(kernel='rbf', n_components=50, random_state=42)
X_nystroem = nystroem.fit_transform(X)

# 2. RBF Sampler
rbf_sampler = RBFSampler(gamma=1.0, n_components=50, random_state=42)
X_rbf_sampler = rbf_sampler.fit_transform(X)

# Vizualizace prvních dvou dimenzí transformovaných dat
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Nystroem
axes[0].scatter(X_nystroem[y==0, 0], X_nystroem[y==0, 1], color='blue', alpha=0.7, label='Třída 0')
axes[0].scatter(X_nystroem[y==1, 0], X_nystroem[y==1, 1], color='red', alpha=0.7, label='Třída 1')
axes[0].set_title('Nystroem Transformace (první 2 dimenze)', fontsize=15)
axes[0].set_xlabel('Příznak 1', fontsize=12)
axes[0].set_ylabel('Příznak 2', fontsize=12)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# RBF Sampler
axes[1].scatter(X_rbf_sampler[y==0, 0], X_rbf_sampler[y==0, 1], color='blue', alpha=0.7, label='Třída 0')
axes[1].scatter(X_rbf_sampler[y==1, 0], X_rbf_sampler[y==1, 1], color='red', alpha=0.7, label='Třída 1')
axes[1].set_title('RBF Sampler Transformace (první 2 dimenze)', fontsize=15)
axes[1].set_xlabel('Příznak 1', fontsize=12)
axes[1].set_ylabel('Příznak 2', fontsize=12)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Zobrazíme dimenzionalitu transformovaných dat
print(f"Původní data: {X.shape}")
print(f"Nystroem transformace: {X_nystroem.shape}")
print(f"RBF Sampler transformace: {X_rbf_sampler.shape}")

## 2. Detailní pohled na Nystroem metodu

Nystroem je metoda pro aproximaci jádrových matic. Pracuje tak, že vybere podmnožinu vzorků a použije je k aproximaci celé jádrové matice.

### Jak Nystroem funguje:

1. Vybere podmnožinu m vzorků z n celkových vzorků (kde m << n)
2. Vypočítá jádrovou matici K pro tyto m vzorků
3. Provede dekompozici vlastních hodnot matice K
4. Použije tuto dekompozici k aproximaci celé jádrové matice

Nyní prozkoumáme Nystroem metodu podrobněji a ukážeme, jak různé parametry ovlivňují její výkon.

In [ ]:
# Vytvoříme dataset měsíců (moons) pro ilustraci
X_moons, y_moons = make_moons(n_samples=1000, noise=0.1, random_state=42)

# Vizualizace původních dat
plt.figure(figsize=(10, 8))
plt.scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], color='blue', alpha=0.7, label='Třída 0')
plt.scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], color='red', alpha=0.7, label='Třída 1')
plt.title('Původní data - dva měsíce', fontsize=15)
plt.xlabel('Příznak 1', fontsize=12)
plt.ylabel('Příznak 2', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Experimentujeme s různým počtem komponent v Nystroem
n_components_list = [2, 10, 50, 100, 200]

# Pro uložení výsledků
nystroem_results = {}

for n_comp in n_components_list:
    # Měříme čas výpočtu
    start_time = time.time()
    
    # Aplikace Nystroem
    nystroem = Nystroem(kernel='rbf', gamma=1.0, n_components=n_comp, random_state=42)
    X_transformed = nystroem.fit_transform(X_moons)
    
    # Uložíme výsledky a čas
    nystroem_results[n_comp] = {
        'transformed_data': X_transformed,
        'computation_time': time.time() - start_time
    }
    
    print(f"Nystroem s {n_comp} komponentami: {nystroem_results[n_comp]['computation_time']:.4f} sekund")

# Vizualizace výsledků pro různý počet komponent
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Původní data
axes[0].scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], color='blue', alpha=0.7, label='Třída 0')
axes[0].scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], color='red', alpha=0.7, label='Třída 1')
axes[0].set_title('Původní data', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Výsledky pro různý počet komponent
for i, n_comp in enumerate(n_components_list):
    axes[i+1].scatter(
        nystroem_results[n_comp]['transformed_data'][y_moons==0, 0],
        nystroem_results[n_comp]['transformed_data'][y_moons==0, 1],
        color='blue', alpha=0.7, label='Třída 0'
    )
    axes[i+1].scatter(
        nystroem_results[n_comp]['transformed_data'][y_moons==1, 0],
        nystroem_results[n_comp]['transformed_data'][y_moons==1, 1],
        color='red', alpha=0.7, label='Třída 1'
    )
    axes[i+1].set_title(f'Nystroem - {n_comp} komponent ({nystroem_results[n_comp]["computation_time"]:.3f}s)', fontsize=14)
    axes[i+1].legend(fontsize=10)
    axes[i+1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Vliv parametru kernel na Nystroem transformaci
kernels = ['rbf', 'poly', 'sigmoid', 'cosine']
kernel_results = {}

for kernel in kernels:
    # Měříme čas výpočtu
    start_time = time.time()
    
    # Aplikace Nystroem s různými jádry
    nystroem = Nystroem(kernel=kernel, n_components=50, random_state=42)
    try:
        X_transformed = nystroem.fit_transform(X_moons)
        
        # Uložíme výsledky a čas
        kernel_results[kernel] = {
            'transformed_data': X_transformed,
            'computation_time': time.time() - start_time,
            'error': None
        }
        
        print(f"Nystroem s jádrem '{kernel}': {kernel_results[kernel]['computation_time']:.4f} sekund")
    except Exception as e:
        kernel_results[kernel] = {
            'transformed_data': None,
            'computation_time': None,
            'error': str(e)
        }
        print(f"Chyba při použití jádra '{kernel}': {e}")

# Vizualizace výsledků pro různá jádra
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for i, kernel in enumerate(kernels):
    if kernel_results[kernel]['transformed_data'] is not None:
        axes[i].scatter(
            kernel_results[kernel]['transformed_data'][y_moons==0, 0],
            kernel_results[kernel]['transformed_data'][y_moons==0, 1],
            color='blue', alpha=0.7, label='Třída 0'
        )
        axes[i].scatter(
            kernel_results[kernel]['transformed_data'][y_moons==1, 0],
            kernel_results[kernel]['transformed_data'][y_moons==1, 1],
            color='red', alpha=0.7, label='Třída 1'
        )
        axes[i].set_title(f'Nystroem - jádro {kernel} ({kernel_results[kernel]["computation_time"]:.3f}s)', fontsize=14)
    else:
        axes[i].text(0.5, 0.5, f"Chyba: {kernel_results[kernel]['error']}", 
                    ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(f'Nystroem - jádro {kernel} (chyba)', fontsize=14)
    
    axes[i].legend(fontsize=10)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Porovnání RBFSampler s Nystroem metodou

RBFSampler je metoda pro aproximaci RBF jádra pomocí náhodných Fourierových vlastností. Nyní porovnáme RBFSampler s Nystroem metodou z hlediska výkonu a efektivity.

In [ ]:
# Vytvoření většího datasetu pro testování efektivity
n_samples = 5000
X_large, y_large = make_circles(n_samples=n_samples, factor=0.3, noise=0.05, random_state=42)

# Seznam různých počtů komponent pro testování
n_components_list = [10, 50, 100, 200, 500]

# Pro uložení výsledků
comparison_results = {'nystroem': [], 'rbf_sampler': []}

# Testování pro různý počet komponent
for n_comp in n_components_list:
    # Nystroem
    start_time = time.time()
    nystroem = Nystroem(kernel='rbf', n_components=n_comp, random_state=42)
    X_nyst = nystroem.fit_transform(X_large)
    nystroem_time = time.time() - start_time
    
    # RBFSampler
    start_time = time.time()
    rbf_sampler = RBFSampler(gamma=1.0, n_components=n_comp, random_state=42)
    X_rbf = rbf_sampler.fit_transform(X_large)
    rbf_time = time.time() - start_time
    
    # Uložení výsledků
    comparison_results['nystroem'].append({
        'n_components': n_comp,
        'time': nystroem_time,
        'transformed_data': X_nyst,
        'shape': X_nyst.shape
    })
    
    comparison_results['rbf_sampler'].append({
        'n_components': n_comp,
        'time': rbf_time,
        'transformed_data': X_rbf,
        'shape': X_rbf.shape
    })
    
    print(f"n_components={n_comp}:")
    print(f"  Nystroem: {nystroem_time:.4f} sekund")
    print(f"  RBFSampler: {rbf_time:.4f} sekund")
    print(f"  Poměr: Nystroem je {nystroem_time/rbf_time:.2f}x pomalejší než RBFSampler")

# Vizualizace časů výpočtů
plt.figure(figsize=(12, 6))

plt.plot(
    [r['n_components'] for r in comparison_results['nystroem']], 
    [r['time'] for r in comparison_results['nystroem']], 
    'o-', linewidth=2, label='Nystroem'
)
plt.plot(
    [r['n_components'] for r in comparison_results['rbf_sampler']], 
    [r['time'] for r in comparison_results['rbf_sampler']], 
    's-', linewidth=2, label='RBF Sampler'
)

plt.xlabel('Počet komponent', fontsize=14)
plt.ylabel('Čas výpočtu (sekundy)', fontsize=14)
plt.title('Porovnání rychlosti Nystroem a RBF Sampler', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Porovnáme přesnost Nystroem a RBFSampler pro klasifikaci

# Rozdělení dat na trénovací a testovací sadu
X_train, X_test, y_train, y_test = train_test_split(
    X_large, y_large, test_size=0.3, random_state=42)

# Standardizace dat
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Experimenty pro různý počet komponent
n_comp_test = [10, 50, 100, 200, 500]
classification_results = {'nystroem': [], 'rbf_sampler': [], 'svc': None}

# Baseline - nelineární SVC s RBF jádrem
start_time = time.time()
svc = SVC(kernel='rbf', gamma='scale')
svc.fit(X_train_scaled, y_train)
svc_time = time.time() - start_time
svc_accuracy = accuracy_score(y_test, svc.predict(X_test_scaled))
classification_results['svc'] = {'time': svc_time, 'accuracy': svc_accuracy}
print(f"SVC s RBF jádrem: čas={svc_time:.4f}s, přesnost={svc_accuracy:.4f}")

# Pro každý počet komponent
for n_comp in n_comp_test:
    # Nystroem + LinearSVC
    start_time = time.time()
    nystroem = Nystroem(kernel='rbf', n_components=n_comp, random_state=42)
    X_train_nyst = nystroem.fit_transform(X_train_scaled)
    X_test_nyst = nystroem.transform(X_test_scaled)
    
    linear_svc = LinearSVC()
    linear_svc.fit(X_train_nyst, y_train)
    nystroem_time = time.time() - start_time
    nystroem_accuracy = accuracy_score(y_test, linear_svc.predict(X_test_nyst))
    
    # RBFSampler + LinearSVC
    start_time = time.time()
    rbf_sampler = RBFSampler(gamma=1.0, n_components=n_comp, random_state=42)
    X_train_rbf = rbf_sampler.fit_transform(X_train_scaled)
    X_test_rbf = rbf_sampler.transform(X_test_scaled)
    
    linear_svc = LinearSVC()
    linear_svc.fit(X_train_rbf, y_train)
    rbf_time = time.time() - start_time
    rbf_accuracy = accuracy_score(y_test, linear_svc.predict(X_test_rbf))
    
    # Uložení výsledků
    classification_results['nystroem'].append({
        'n_components': n_comp,
        'time': nystroem_time,
        'accuracy': nystroem_accuracy
    })
    
    classification_results['rbf_sampler'].append({
        'n_components': n_comp,
        'time': rbf_time,
        'accuracy': rbf_accuracy
    })
    
    print(f"n_components={n_comp}:")
    print(f"  Nystroem + LinearSVC: čas={nystroem_time:.4f}s, přesnost={nystroem_accuracy:.4f}")
    print(f"  RBFSampler + LinearSVC: čas={rbf_time:.4f}s, přesnost={rbf_accuracy:.4f}")

# Vizualizace výsledků přesnosti
plt.figure(figsize=(12, 6))

plt.plot(
    [r['n_components'] for r in classification_results['nystroem']], 
    [r['accuracy'] for r in classification_results['nystroem']], 
    'o-', linewidth=2, label='Nystroem + LinearSVC'
)
plt.plot(
    [r['n_components'] for r in classification_results['rbf_sampler']], 
    [r['accuracy'] for r in classification_results['rbf_sampler']], 
    's-', linewidth=2, label='RBF Sampler + LinearSVC'
)
plt.axhline(y=classification_results['svc']['accuracy'], color='r', linestyle='--', 
           label=f'SVC s RBF jádrem (baseline): {classification_results["svc"]["accuracy"]:.4f}')

plt.xlabel('Počet komponent', fontsize=14)
plt.ylabel('Přesnost', fontsize=14)
plt.title('Porovnání přesnosti klasifikace', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

# Vizualizace času výpočtu
plt.figure(figsize=(12, 6))

plt.plot(
    [r['n_components'] for r in classification_results['nystroem']], 
    [r['time'] for r in classification_results['nystroem']], 
    'o-', linewidth=2, label='Nystroem + LinearSVC'
)
plt.plot(
    [r['n_components'] for r in classification_results['rbf_sampler']], 
    [r['time'] for r in classification_results['rbf_sampler']], 
    's-', linewidth=2, label='RBF Sampler + LinearSVC'
)
plt.axhline(y=classification_results['svc']['time'], color='r', linestyle='--', 
           label=f'SVC s RBF jádrem (baseline): {classification_results["svc"]["time"]:.4f}s')

plt.xlabel('Počet komponent', fontsize=14)
plt.ylabel('Čas (sekundy)', fontsize=14)
plt.title('Porovnání času klasifikace', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 4. Kernel Approximation pro redukci dimenzionality

Nyní se zaměříme na použití kernel approximation pro redukci dimenzionality. Porovnáme jej s jinými metodami redukce dimenzionality, jako je PCA, t-SNE a Isomap.

In [ ]:
# Načteme dataset MNIST číslic pro příklad redukce dimenzionality
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Tvar dat MNIST: {X_digits.shape}")
print(f"Počet tříd: {len(np.unique(y_digits))}")

# Zobrazení několika příkladů číslic
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Číslice: {y_digits[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Standardizace dat
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

# Aplikujeme různé metody redukce dimenzionality
methods = {
    'PCA': PCA(n_components=2, random_state=42),
    'Nystroem': Nystroem(kernel='rbf', n_components=2, random_state=42),
    'RBF Sampler': RBFSampler(gamma=0.01, n_components=2, random_state=42),
    't-SNE': TSNE(n_components=2, random_state=42),
    'Isomap': Isomap(n_components=2, n_neighbors=5)
}

# Aplikace metod a měření času
results = {}
for name, method in methods.items():
    print(f"Aplikuji metodu: {name}...")
    start_time = time.time()
    X_reduced = method.fit_transform(X_digits_scaled)
    computation_time = time.time() - start_time
    results[name] = {
        'reduced_data': X_reduced,
        'time': computation_time
    }
    print(f"  Čas výpočtu: {computation_time:.4f} sekund")

# Vizualizace výsledků redukce dimenzionality
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (name, result) in enumerate(results.items()):
    scatter = axes[i].scatter(
        result['reduced_data'][:, 0], 
        result['reduced_data'][:, 1], 
        c=y_digits, 
        cmap='tab10', 
        alpha=0.7,
        s=30
    )
    axes[i].set_title(f'{name} ({result["time"]:.2f}s)', fontsize=14)
    axes[i].grid(True, alpha=0.3)
    
# Odstraníme prázdný subplor
if len(results) < len(axes):
    fig.delaxes(axes[-1])
    
# Přidáme legendu
plt.colorbar(scatter, ax=axes, label='Třída (číslice)')
plt.tight_layout()
plt.show()

## 5. Porovnání zachování podobnosti pro redukci dimenzionality

Důležitým aspektem metod redukce dimenzionality je, jak dobře zachovávají podobnosti mezi datovými body. Nyní porovnáme, jak dobře různé metody (včetně kernel approximation) zachovávají podobnosti pomocí korelace mezi původními a redukovanými vzdálenostmi.

In [ ]:
# Pro tuto analýzu použijeme menší množinu dat pro rychlejší výpočet
n_samples_subset = 500
indices = np.random.choice(len(X_digits_scaled), n_samples_subset, replace=False)
X_subset = X_digits_scaled[indices]
y_subset = y_digits[indices]

# Vypočteme matici vzdáleností v původním prostoru
original_distances = pairwise_distances(X_subset)

# Aplikujeme metody redukce dimenzionality na podmnožinu
subset_results = {}
for name, method in methods.items():
    print(f"Aplikuji metodu: {name} na podmnožinu...")
    start_time = time.time()
    X_reduced = method.fit_transform(X_subset)
    computation_time = time.time() - start_time
    
    # Výpočet vzdáleností v redukovaném prostoru
    reduced_distances = pairwise_distances(X_reduced)
    
    # Výpočet korelace mezi původními a redukovanými vzdálenostmi
    # Použijeme pouze horní trojúhelník matice (bez diagonály)
    triu_indices = np.triu_indices(n_samples_subset, k=1)
    correlation = np.corrcoef(original_distances[triu_indices], reduced_distances[triu_indices])[0, 1]
    
    subset_results[name] = {
        'reduced_data': X_reduced,
        'time': computation_time,
        'correlation': correlation
    }
    print(f"  Čas výpočtu: {computation_time:.4f} sekund")
    print(f"  Korelace vzdáleností: {correlation:.4f}")

# Vizualizace výsledků korelace
plt.figure(figsize=(12, 6))
correlation_values = [result['correlation'] for result in subset_results.values()]
method_names = list(subset_results.keys())

bars = plt.bar(method_names, correlation_values)
for i, (name, result) in enumerate(subset_results.items()):
    plt.text(i, result['correlation'] + 0.02, 
            f"{result['correlation']:.3f}", 
            ha='center', fontsize=10)

plt.xlabel('Metoda redukce dimenzionality', fontsize=14)
plt.ylabel('Korelace vzdáleností', fontsize=14)
plt.title('Porovnání zachování podobností mezi datovými body', fontsize=16)
plt.ylim(0, 1.0)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Analýza škálování s velikostí datasetu

Prozkoumejme, jak se různé metody kernel approximation škálují s rostoucím počtem vzorků a s rostoucí dimenzionalitou dat.

In [ ]:
# Testování škálování s počtem vzorků
sample_sizes = [100, 500, 1000, 2000, 5000, 10000]
features = 20  # Fixní počet příznaků

# Metody, které budeme testovat
scaling_methods = {
    'Nystroem': lambda: Nystroem(n_components=100, random_state=42),
    'RBF Sampler': lambda: RBFSampler(n_components=100, random_state=42),
    'SkewedChi2Sampler': lambda: SkewedChi2Sampler(n_components=100, random_state=42),
    'PCA': lambda: PCA(n_components=2, random_state=42)
}

# Uložení časů
times = {name: [] for name in scaling_methods}

for n_samples in sample_sizes:
    # Vytvoření náhodných dat
    np.random.seed(42)
    X = np.random.randn(n_samples, features)
    
    print(f"Počet vzorků: {n_samples}")
    for name, method_fn in scaling_methods.items():
        # Vytvoření nové instance metody
        method = method_fn()
        
        # Měření času výpočtu
        start_time = time.time()
        try:
            method.fit_transform(X)
            elapsed = time.time() - start_time
            times[name].append(elapsed)
            print(f"  {name}: {elapsed:.4f} sekund")
        except Exception as e:
            print(f"  {name}: Chyba - {e}")
            times[name].append(np.nan)

# Vizualizace výsledků škálování s počtem vzorků
plt.figure(figsize=(12, 6))

for name, time_values in times.items():
    plt.plot(sample_sizes, time_values, 'o-', linewidth=2, label=name)
    
plt.xlabel('Počet vzorků', fontsize=14)
plt.ylabel('Čas výpočtu (sekundy)', fontsize=14)
plt.title('Škálování metod s počtem vzorků', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.yscale('log')
plt.show()

In [ ]:
# Testování škálování s počtem příznaků
n_samples = 1000  # Fixní počet vzorků
feature_sizes = [10, 50, 100, 500, 1000, 2000]

# Uložení časů
feature_times = {name: [] for name in scaling_methods}

for n_features in feature_sizes:
    # Vytvoření náhodných dat
    np.random.seed(42)
    X = np.random.randn(n_samples, n_features)
    
    print(f"Počet příznaků: {n_features}")
    for name, method_fn in scaling_methods.items():
        # Vytvoření nové instance metody
        method = method_fn()
        
        # Měření času výpočtu
        start_time = time.time()
        try:
            method.fit_transform(X)
            elapsed = time.time() - start_time
            feature_times[name].append(elapsed)
            print(f"  {name}: {elapsed:.4f} sekund")
        except Exception as e:
            print(f"  {name}: Chyba - {e}")
            feature_times[name].append(np.nan)

# Vizualizace výsledků škálování s počtem příznaků
plt.figure(figsize=(12, 6))

for name, time_values in feature_times.items():
    plt.plot(feature_sizes, time_values, 'o-', linewidth=2, label=name)
    
plt.xlabel('Počet příznaků', fontsize=14)
plt.ylabel('Čas výpočtu (sekundy)', fontsize=14)
plt.title('Škálování metod s počtem příznaků', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.tight_layout()
plt.yscale('log')
plt.show()

## 7. Použití Kernel Approximation v Pipeline

Nyní ukážeme, jak používat Kernel Approximation v rámci scikit-learn Pipeline pro předzpracování dat.

In [ ]:
# Vytvoření Swiss Roll datasetu s více přidanými dimenzemi (šumem)
X_swiss, y_swiss = make_swiss_roll(n_samples=2000, noise=0.1, random_state=42)
X_swiss = X_swiss[:, :2]  # Použijeme jen první dvě dimenze pro vizualizaci

# Přidáme šumové dimenze
np.random.seed(42)
noise_dims = np.random.randn(X_swiss.shape[0], 30)  # 30 šumových dimenzí
X_swiss_noisy = np.hstack([X_swiss, noise_dims])  # Celkem 32 dimenzí

print(f"Tvar dat s šumem: {X_swiss_noisy.shape}")

# Vizualizace původních dat (první dvě dimenze)
plt.figure(figsize=(10, 8))
plt.scatter(X_swiss[:, 0], X_swiss[:, 1], c=y_swiss, cmap='viridis', s=30, alpha=0.8)
plt.title('Původní data (první 2 dimenze z 32)', fontsize=15)
plt.colorbar(label='Pozice na spirále')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Vytvoření pipeline s Kernel Approximation
from sklearn.pipeline import Pipeline

# 1. Pipeline s Nystroem
nystroem_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('nystroem', Nystroem(kernel='rbf', n_components=2, random_state=42))
])

# 2. Pipeline s RBFSampler
rbf_sampler_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rbf_sampler', RBFSampler(gamma=0.1, n_components=2, random_state=42))
])

# 3. Pipeline s PCA pro porovnání
pca_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=42))
])

# Aplikace pipeline
pipelines = {
    'Nystroem Pipeline': nystroem_pipeline,
    'RBF Sampler Pipeline': rbf_sampler_pipeline,
    'PCA Pipeline': pca_pipeline
}

pipeline_results = {}
for name, pipeline in pipelines.items():
    start_time = time.time()
    X_transformed = pipeline.fit_transform(X_swiss_noisy)
    pipeline_results[name] = {
        'transformed_data': X_transformed,
        'time': time.time() - start_time
    }
    print(f"{name}: {pipeline_results[name]['time']:.4f} sekund")

# Vizualizace výsledků
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (name, result) in enumerate(pipeline_results.items()):
    axes[i].scatter(
        result['transformed_data'][:, 0], 
        result['transformed_data'][:, 1], 
        c=y_swiss, 
        cmap='viridis', 
        s=30, 
        alpha=0.8
    )
    axes[i].set_title(f'{name} ({result["time"]:.2f}s)', fontsize=14)
    axes[i].grid(True, alpha=0.3)
    
plt.tight_layout()
plt.show()

## 8. Shrnutí a doporučení pro použití Kernel Approximation

### Výhody Kernel Approximation:

1. **Škálovatelnost**: Umožňuje aplikaci jádrových metod na velké datové sady s lineární časovou složitostí
2. **Efektivita paměti**: Nevyžaduje ukládání plné jádrové matice (která má složitost O(n²))
3. **Integrace s lineárními metodami**: Umožňuje použití lineárních modelů s nelineárními transformacemi dat
4. **Paralelizovatelnost**: Mnoho implementací lze snadno paralelizovat
5. **Online učení**: Podporuje inkrementální aktualizace, což je vhodné pro streamování dat

### Nevýhody Kernel Approximation:

1. **Aproximativní výsledky**: Nedosahuje přesnosti plných jádrových metod, zejména při malém počtu komponent
2. **Ladění hyperparametrů**: Může být obtížné najít optimální počet komponent a parametry jádra
3. **Nestabilita**: Některé implementace mohou poskytovat nestabilní výsledky pro různé inicializace
4. **Interpretovatelnost**: Výsledné transformované příznaky nemusí být snadno interpretovatelné
5. **Specifické pro jádro**: Každá metoda je typicky navržena pro konkrétní typ jádrové funkce

### Doporučení pro použití:

- **Kdy použít Nystroem**:
  - Pro střední až velké datové sady
  - Když potřebujete přesnou aproximaci jádrové matice
  - Když váš problém těží z přesnosti více než z rychlosti
  - Flexibilní volba jádrové funkce (rbf, poly, sigmoid atd.)

- **Kdy použít RBFSampler**:
  - Pro velmi velké datové sady
  - Když je rychlost důležitější než absolutní přesnost
  - Specificky pro aproximaci RBF jádra
  - Když potřebujete velmi rychlé výpočty

- **Optimální nastavení parametrů**:
  - `n_components`: Větší hodnoty poskytují přesnější aproximaci, ale zvyšují výpočetní nároky (typicky 50-500)
  - `gamma` (pro RBF jádro): Ovlivňuje šířku Gaussovy křivky, menší hodnoty vytváří hladší transformace
  - Pro Nystroem zvažte různé typy jader podle charakteru vašich dat

### Závěr:

Kernel Approximation poskytuje efektivní způsob, jak využít sílu jádrových metod bez prohibitivních výpočetních nároků plných jádrových matic. Při vhodném nastavení může dosáhnout výsledků blížících se plným jádrovým metodám při zlomku výpočetních nároků, což je obzvláště cenné pro velké datové sady. Pro redukci dimenzionality nabízí zajímavý kompromis mezi lineárními metodami jako PCA a náročnějšími nelineárními metodami jako t-SNE nebo Isomap.